In [ ]:
from __future__ import annotations

import csv
import os
import re
import shutil
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable, Literal

import numpy as np


def resolve_repository_root() -> Path:
    root = Path(
        os.environ.get(
            "PHYLOGENY_REPOSITORY_ROOT",
            Path.cwd(),
        )
    ).expanduser().resolve()

    if root.name in {"AA", "NT", "common"}:
        root = root.parent

    return root


REPOSITORY_ROOT = resolve_repository_root()
NT_PROJECT_ROOT = REPOSITORY_ROOT / "NT"
IQTREE_BIN_DIR = (
    Path(os.environ["IQTREE_BIN_DIR"]).expanduser().resolve()
    if os.environ.get("IQTREE_BIN_DIR")
    else None
)


GeneratorType = Literal["rtree", "yule"]


@dataclass(frozen=True)
class SimConfig:
    # Project root for nucleotide experiment
    project_root: Path = NT_PROJECT_ROOT

    out_root: Path = NT_PROJECT_ROOT / "data"
    iqtree_bin_dir: Path | None = IQTREE_BIN_DIR

    generators: tuple[GeneratorType, ...] = ("rtree", "yule")
    n_taxa_list: tuple[int, ...] = (30,)
    mean_branch_lengths: tuple[float, ...] = (0.125, 0.250, 0.500, 0.625, 0.750)

    n_reps: int = 100

    seqtype: str = "DNA"
    model: str = "JC+G5"
    seq_length: int = 500

    global_seed: int = 20260416
    overwrite: bool = False


@dataclass
class ManifestRow:
    tag: str
    generator: str
    n_taxa: int
    target_mean_branch_length: float

    seqtype: str
    model: str
    seq_length: int

    tree_seed: int
    seq_seed: int
    tree_path: str
    fasta_path: str
    log_path: str
    observed_model: str
    observed_rate_heterogeneity: str
    observed_gamma_shape: str
    status: str
    message: str


def ensure_dirs(cfg: SimConfig) -> tuple[Path, Path, Path]:
    trees_dir = cfg.out_root / "trees"
    sim_dir = cfg.out_root / "sim"
    mani_dir = cfg.out_root / "manifests"

    trees_dir.mkdir(parents=True, exist_ok=True)
    sim_dir.mkdir(parents=True, exist_ok=True)
    mani_dir.mkdir(parents=True, exist_ok=True)

    return trees_dir, sim_dir, mani_dir


def set_iqtree_path(cfg: SimConfig) -> None:
    if cfg.iqtree_bin_dir is not None:
        os.environ["PATH"] = (
            str(cfg.iqtree_bin_dir)
            + os.pathsep
            + os.environ["PATH"]
        )

    if shutil.which("iqtree3") is None:
        raise FileNotFoundError(
            "iqtree3 was not found. Add it to PATH or set IQTREE_BIN_DIR."
        )


def make_tag(generator: str, n_taxa: int, mean_bl: float, rep: int) -> str:
    # Keep the same tag format as the AA pipeline.
    # The sequence types are separated by their project directories.
    return f"{generator}_n{n_taxa}_bl{mean_bl:.3f}_rep{rep:03d}"


def write_r_script(script_path: Path) -> None:
    """
    Tree generation is delegated to R, following the existing AA-side pipeline.
    """
    script = r'''
args <- commandArgs(trailingOnly = TRUE)
out_csv <- args[1]

suppressPackageStartupMessages(library(ape))
suppressPackageStartupMessages(library(TreeSim))

df <- read.csv(out_csv, stringsAsFactors = FALSE)

for (i in seq_len(nrow(df))) {
  tag <- df$tag[i]
  generator <- df$generator[i]
  n_taxa <- df$n_taxa[i]
  mean_bl <- df$target_mean_branch_length[i]
  tree_seed <- df$tree_seed[i]
  tree_path <- df$tree_path[i]

  set.seed(tree_seed)

  if (generator == "rtree") {
    tr <- rtree(n = n_taxa)

    # Match the target mean branch length by drawing branch lengths
    # from an exponential distribution with the specified mean.
    E <- nrow(tr$edge)
    bl <- rexp(E, rate = 1.0 / mean_bl)
    bl <- pmax(bl, 1e-6)
    tr$edge.length <- bl

  } else if (generator == "yule") {
    tr <- sim.bd.taxa(n = n_taxa, numbsim = 1, lambda = 1, mu = 0)[[1]]

    current_mean <- mean(tr$edge.length)
    tr$edge.length <- tr$edge.length * (mean_bl / current_mean)

  } else {
    stop(paste("unknown generator:", generator))
  }

  write.tree(tr, file = tree_path)
}
'''
    script_path.write_text(script)


def build_tree_plan(cfg: SimConfig, trees_dir: Path, sim_dir: Path) -> list[ManifestRow]:
    rows: list[ManifestRow] = []
    rng = np.random.default_rng(cfg.global_seed)

    for generator in cfg.generators:
        for n_taxa in cfg.n_taxa_list:
            for mean_bl in cfg.mean_branch_lengths:
                for rep in range(1, cfg.n_reps + 1):
                    tag = make_tag(generator, n_taxa, mean_bl, rep)

                    tree_seed = int(rng.integers(1, 2**31 - 1))
                    seq_seed = int(rng.integers(1, 2**31 - 1))

                    tree_path = trees_dir / f"{tag}.nwk"
                    fasta_path = sim_dir / f"{tag}.fa"
                    log_path = sim_dir / f"{tag}.log"

                    rows.append(
                        ManifestRow(
                            tag=tag,
                            generator=generator,
                            n_taxa=n_taxa,
                            target_mean_branch_length=float(mean_bl),
                            seqtype=cfg.seqtype,
                            model=cfg.model,
                            seq_length=cfg.seq_length,
                            tree_seed=tree_seed,
                            seq_seed=seq_seed,
                            tree_path=str(tree_path),
                            fasta_path=str(fasta_path),
                            log_path=str(log_path),
                            observed_model="",
                            observed_rate_heterogeneity="",
                            observed_gamma_shape="",
                            status="planned",
                            message="",
                        )
                    )

    return rows


def save_manifest(
    rows: list[ManifestRow],
    manifest_path: Path,
    *,
    relative_to: Path | None = None,
) -> None:
    fieldnames = (
        list(asdict(rows[0]).keys())
        if rows
        else list(ManifestRow.__annotations__.keys())
    )
    path_fields = {
        "tree_path",
        "fasta_path",
        "log_path",
    }

    root = (
        relative_to.expanduser().resolve()
        if relative_to is not None
        else None
    )

    with manifest_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in rows:
            record = asdict(row)

            if root is not None:
                for field in path_fields:
                    value = record.get(field)

                    if not value:
                        continue

                    path = Path(value).expanduser().resolve()

                    try:
                        record[field] = (
                            path.relative_to(root).as_posix()
                        )
                    except ValueError as error:
                        raise ValueError(
                            f"{field} is outside project root: {path}"
                        ) from error

            writer.writerow(record)


def run_r_tree_generation(plan_csv: Path, r_script_path: Path) -> None:
    cmd = ["Rscript", str(r_script_path), str(plan_csv)]
    subprocess.run(cmd, check=True)


def generate_reference_trees(
    rows: list[ManifestRow],
    cfg: SimConfig,
    manifest_dir: Path,
) -> None:
    pending_rows = [
        row
        for row in rows
        if cfg.overwrite or not Path(row.tree_path).exists()
    ]

    if not pending_rows:
        print("Reference trees: all files already exist.")
        return

    plan_csv = manifest_dir / "pending_tree_generation_plan.csv"
    r_script = manifest_dir / "generate_trees.R"

    save_manifest(
        pending_rows,
        plan_csv,
    )
    write_r_script(r_script)
    run_r_tree_generation(plan_csv, r_script)

    print(f"Reference trees generated: {len(pending_rows)}")


def parse_alisim_log(log_path: Path) -> tuple[str, str, str]:
    if not log_path.exists():
        return "", "", ""

    text = log_path.read_text(errors="ignore")

    model_match = re.search(r"Model of substitution:\s*(.+)", text)
    rate_match = re.search(r"Model of rate heterogeneity:\s*(.+)", text)
    gamma_match = re.search(r"Gamma shape alpha:\s*([0-9.]+)", text)

    observed_model = model_match.group(1).strip() if model_match else ""
    observed_rate = rate_match.group(1).strip() if rate_match else ""
    observed_gamma = gamma_match.group(1).strip() if gamma_match else ""

    return observed_model, observed_rate, observed_gamma


def run_alisim_for_row(row: ManifestRow, cfg: SimConfig) -> ManifestRow:
    base = Path(row.fasta_path).with_suffix("")

    if Path(row.fasta_path).exists() and not cfg.overwrite:
        observed_model, observed_rate, observed_gamma = parse_alisim_log(Path(row.log_path))

        return ManifestRow(
            **{
                **asdict(row),
                "observed_model": observed_model,
                "observed_rate_heterogeneity": observed_rate,
                "observed_gamma_shape": observed_gamma,
                "status": "ok",
                "message": "skipped_existing",
            }
        )

    if not Path(row.tree_path).exists():
        return ManifestRow(
            **{
                **asdict(row),
                "status": "failed",
                "message": f"Reference tree was not found: {row.tree_path}",
            }
        )

    cmd = [
        "iqtree3",
        "--alisim", str(base),
        "--tree", row.tree_path,
        "--seqtype", row.seqtype,
        "-m", row.model,
        "--length", str(row.seq_length),
        "--seed", str(row.seq_seed),
        "-af", "fasta",
    ]

    if cfg.overwrite:
        cmd.append("--redo")

    try:
        subprocess.run(cmd, check=True, capture_output=True, text=True)

        if not Path(row.fasta_path).exists():
            return ManifestRow(
                **{
                    **asdict(row),
                    "status": "failed",
                    "message": "AliSim finished, but FASTA was not found.",
                }
            )

        observed_model, observed_rate, observed_gamma = parse_alisim_log(Path(row.log_path))

        return ManifestRow(
            **{
                **asdict(row),
                "observed_model": observed_model,
                "observed_rate_heterogeneity": observed_rate,
                "observed_gamma_shape": observed_gamma,
                "status": "ok",
                "message": "",
            }
        )

    except subprocess.CalledProcessError as e:
        return ManifestRow(
            **{
                **asdict(row),
                "status": "failed",
                "message": (e.stderr or e.stdout or str(e))[:2000],
            }
        )


def main() -> None:
    cfg = SimConfig()

    cfg.project_root.mkdir(parents=True, exist_ok=True)
    os.chdir(cfg.project_root)

    trees_dir, sim_dir, mani_dir = ensure_dirs(cfg)
    set_iqtree_path(cfg)

    plan_rows = build_tree_plan(cfg, trees_dir, sim_dir)

    plan_csv = mani_dir / "simulation_plan.csv"
    manifest_csv = mani_dir / "simulation_manifest.csv"

    # Internal execution plan: keep absolute paths for R and AliSim.
    save_manifest(plan_rows, plan_csv)

    generate_reference_trees(
        rows=plan_rows,
        cfg=cfg,
        manifest_dir=mani_dir,
    )

    done_rows: list[ManifestRow] = []

    for row in plan_rows:
        print(f"[process] {row.tag}")
        done_rows.append(run_alisim_for_row(row, cfg))

    # Released manifest: store paths relative to NT/.
    save_manifest(
        done_rows,
        manifest_csv,
        relative_to=cfg.project_root,
    )
    print(f"Saved manifest: {manifest_csv}")


if __name__ == "__main__":
    main()
